In [17]:
# =====================================================================
# SEL 1: IMPORT LIBRARY & KONFIGURASI PATH
# =====================================================================
import os
import joblib
import mlflow
from mlflow.tracking import MlflowClient

# Folder tempat menyimpan file fisik .pkl untuk produksi API
models_out_dir = os.path.join("..", "models", "saved_models")
os.makedirs(models_out_dir, exist_ok=True)

# Arahkan MLflow ke folder lokal mlruns
mlflow.set_tracking_uri("file:./mlruns")
client = MlflowClient()

print(f"✅ Direktori output siap: {os.path.abspath(models_out_dir)}")

✅ Direktori output siap: d:\lentera-laut\src\models\saved_models


In [ ]:
# =====================================================================
# SEL 2: KONFIGURASI PETA MODEL FINAL TERBAIK 
# =====================================================================
# Pemetaan diperbarui dengan menambahkan argumen ketiga (nama folder artefak)
# Format: "target_variable": ("Nama_Eksperimen", "Nama_Run_MLflow", "Folder_Artefak")

final_model_map = {
    # Artefak Baseline tersimpan di folder "model"
    "wave_height": ("LenteraLaut_v3_wave_height", "Baseline_LinearRegression", "model"),
    "ocean_current_velocity": ("LenteraLaut_v3_ocean_current_velocity", "Baseline_ExtraTreesRegressor", "model"),
    "visibility": ("LenteraLaut_v3_visibility", "Baseline_ExtraTreesRegressor", "model"),
    
    # Artefak Optuna tersimpan di folder "tuned_model"
    "sea_surface_temperature": ("LenteraLaut_v4_Tuned_sea_surface_temperature", "Optuna_ExtraTreesRegressor", "tuned_model"),
    "wind_speed_10m": ("LenteraLaut_v4_Tuned_wind_speed_10m", "Optuna_RandomForestRegressor", "tuned_model"),
    "precipitation": ("LenteraLaut_v4_Tuned_precipitation", "Optuna_LGBMRegressor", "tuned_model")
}

print("✅ Konfigurasi pemetaan model juara (beserta jalurnya) berhasil diperbarui.")

✅ Konfigurasi pemetaan model juara (beserta jalurnya) berhasil diperbarui.


In [21]:
# =====================================================================
# SEL 3: EKSTRAKSI DARI MLFLOW DAN SIMPAN KE .PKL (DIUPDATE)
# =====================================================================
import warnings
warnings.filterwarnings("ignore")

print("--- MEMULAI EKSTRAKSI MODEL KE BENTUK FISIK ---")

for target, (exp_name, run_name, artifact_path) in final_model_map.items():
    print(f"\nMencari model untuk: {target}")
    
    # 1. Cari ID Eksperimen
    experiment = client.get_experiment_by_name(exp_name)
    if experiment is None:
        print(f"    ❌ GAGAL: Eksperimen '{exp_name}' tidak ditemukan.")
        continue
        
    # 2. Cari Run yang spesifik
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'"
    )
    
    if not runs:
        print(f"    ❌ GAGAL: Run '{run_name}' tidak ditemukan.")
        continue
        
    best_run_id = runs[0].info.run_id
    
    try:
        # 3. Load model menggunakan nama folder artefak dinamis (artifact_path)
        model_uri = f"runs:/{best_run_id}/{artifact_path}"
        loaded_model = mlflow.sklearn.load_model(model_uri)
        
        # 4. Simpan ke fisik
        file_name = f"model_{target}.pkl"
        file_path = os.path.join(models_out_dir, file_name)
        
        joblib.dump(loaded_model, file_path)
        print(f"    ✅ SUKSES: Model tersimpan secara fisik di {file_name}")
        
    except Exception as e:
        print(f"    ❌ GAGAL saat memuat atau menyimpan model: {e}")

print("\n--- PROSES EKSTRAKSI SELESAI ---")

--- MEMULAI EKSTRAKSI MODEL KE BENTUK FISIK ---

Mencari model untuk: wave_height


    ✅ SUKSES: Model tersimpan secara fisik di model_wave_height.pkl

Mencari model untuk: ocean_current_velocity


    ✅ SUKSES: Model tersimpan secara fisik di model_ocean_current_velocity.pkl

Mencari model untuk: visibility


    ✅ SUKSES: Model tersimpan secara fisik di model_visibility.pkl

Mencari model untuk: sea_surface_temperature


    ✅ SUKSES: Model tersimpan secara fisik di model_sea_surface_temperature.pkl

Mencari model untuk: wind_speed_10m


    ✅ SUKSES: Model tersimpan secara fisik di model_wind_speed_10m.pkl

Mencari model untuk: precipitation


    ✅ SUKSES: Model tersimpan secara fisik di model_precipitation.pkl

--- PROSES EKSTRAKSI SELESAI ---
